In [1]:
import  torch
import torch.nn as nn 
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
import matplotlib.pyplot as plt 


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Device",device)


Device cpu


In [3]:
#Load MNIST

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="data/raw",
    train=True,
    download=True,
    transform=transform

)


test_dataset = datasets.MNIST(
    root="data/raw",
    train=False,
    download=True,
    transform=transform

)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True

)


test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False

)


In [4]:
#Define the ANN

class MNIST_ANN(nn.Module):

    def __init__(self):
        super().__init__()


        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(784,128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128,10)

    def forward(self,x):

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


In [ ]:
#Create the training function

def train_model(model,optimizer,epochs = 5):

    criterion = nn.CrossEntropyLoss()

    history = {
        "loss":[],
        "accuracy":[]
    }


    for epoch in range(epochs):

        model.train()

        total_loss = 0
        correct = 0
        total = 0

        for images,labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            output = model(images)

            #loss
            loss = criterion(output,labels)

            #backpropagation
            loss.backward()

            #update wieghts
            optimizer.step()


            total_loss += loss.item()

            prediction = torch.argmax(
                output,
                dim=1
            )

            correct += (
                prediction == labels
            ).sum().item()

            total += labels.size(0)

        epoch_loss = total_loss / len(train_loader)
        epoch_accuracy = correct / total

        history["loss"].append(epoch_loss)
        history["accuracy"].append(epoch_accuracy)

        print(
            f"Epoch[{epoch+1}/{epochs}]"
            f"Loss:{epoch_loss:.4f}"
            f"Accuracy:{epoch_accuracy:.4f}"
        ) 

    return history


In [ ]:
#Test different learning rates

learning_rate = [
    0.0001,
    0.001,
    0.01
]


In [7]:
import mlflow

mlflow .set_tracking_uri(
    "http://localhost:5000"
)

mlflow.set_experiment(
    "MNIST Learning Rate Comparison"
)


2026/08/11 08:18:13 INFO mlflow.tracking.fluent: Experiment with name 'MNIST Learning Rate Comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1786416493852, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1786416493852, lifecycle_stage='active', name='MNIST Learning Rate Comparison', tags={}, trace_location=None, workspace='default'>

In [8]:
#Then train a fresh model for every learning rate:

histories = {}

for lr in learning_rate:

    print(f"\n====== Leraning Rate:{lr} =====")

    model = MNIST_ANN().to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr

    )


    with mlflow.start_run(
        run_name=f"Adam_lr_{lr}"
    ):

        mlflow.log_param(
            "optimizer",
            "Adam"
        )

        mlflow.log_param(
            "learning_rate",
            lr
        )

        mlflow.log_param(
            "batch_size",
            64
        )

        mlflow.log_param(
            "epochs",
            5
        )

        history = train_model(
            model,
            optimizer,
            epochs=5
        )

        for epoch, accuracy in enumerate(
            history["accuracy"]
        ):

            mlflow.log_metric(
                "train_accuracy",
                accuracy,
                step=epoch
            )





====== Leraning Rate:0.0001 =====
Epoch[1/5]Loss:0.8956Accuracy:0.8124
Epoch[2/5]Loss:0.3723Accuracy:0.9004
Epoch[3/5]Loss:0.3100Accuracy:0.9129
Epoch[4/5]Loss:0.2778Accuracy:0.9213
Epoch[5/5]Loss:0.2546Accuracy:0.9281
🏃 View run Adam_lr_0.0001 at: http://localhost:5000/#/experiments/5/runs/a1b700487a094e48b403a3a35f51b20a
🧪 View experiment at: http://localhost:5000/#/experiments/5

====== Leraning Rate:0.001 =====
Epoch[1/5]Loss:0.3451Accuracy:0.9072
Epoch[2/5]Loss:0.1583Accuracy:0.9540
Epoch[3/5]Loss:0.1109Accuracy:0.9680
Epoch[4/5]Loss:0.0842Accuracy:0.9755
Epoch[5/5]Loss:0.0660Accuracy:0.9803
🏃 View run Adam_lr_0.001 at: http://localhost:5000/#/experiments/5/runs/51dd302bc9284c65a06f5c243bc103c7
🧪 View experiment at: http://localhost:5000/#/experiments/5

====== Leraning Rate:0.01 =====
Epoch[1/5]Loss:0.2194Accuracy:0.9327
Epoch[2/5]Loss:0.1282Accuracy:0.9624
Epoch[3/5]Loss:0.1063Accuracy:0.9693
Epoch[4/5]Loss:0.0998Accuracy:0.9720
Epoch[5/5]Loss:0.0956Accuracy:0.9741
🏃 View run A